In [1]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




[1] "Connected to : yhcr-prd-bradfor-bia-core"


In [2]:
person_ids <- read.csv('data/additional_person_ids.csv', header = TRUE)

## CIN 2009 - 2019

In [3]:
sc_data = "CB_2489.cb_CIN_2009_to_2019"

sc_table <- tbl(con, sc_data) |>
    select(person_id,CIN_ACADYR,CIN_PrimaryNeedCode) 

In [4]:
sc_df <- collect(sc_table)

In [5]:
head(sc_df)

person_id,CIN_ACADYR,CIN_PrimaryNeedCode
<chr>,<chr>,<chr>
60ED8043B497A211AC4A5398BD142426BDA0B3D96FE6AED2698D3DFC02DEEB15,2010/2011,n5
1F9BB6FF8DB3BD8669EB964CE81B338AFB0D26DE652DF93746D396E29177AD01,2018/2019,N1
B159AB8D75C5014034B151D01F7976544F59C57ED857447A87D0486FC65B2028,2018/2019,N8
86DF4B72CC0D700A84163BBA7929456060242ACCDC4A16DFB246E808FEC37CC5,2018/2019,N1
071DA72845B4CFBC9C3DD46124CEAEF73AF06812BE4088B211BDAFF25EA874F4,2018/2019,N1
B577F7CF281A918EF137A9C22D6F0DC9BCF943D97613A5DE816E7136388117B4,2018/2019,N1


In [6]:
sc_df |> arrange(CIN_ACADYR) |> distinct(CIN_ACADYR) |> pull(CIN_ACADYR)

[1] "2008/2009" "2009/2010" "2010/2011" "2011/2012" "2012/2013" "2013/2014"
 [7] "2014/2015" "2015/2016" "2016/2017" "2017/2018" "2018/2019"

In [7]:
sc_df |> distinct(CIN_PrimaryNeedCode) |> pull(CIN_PrimaryNeedCode)

[1] "n5"         "N1        " "N8        " "N4        " NA          
 [6] "N0"         "N1"         "N2"         "N3"         "N4"        
[11] "N5"         "N6"         "N7"         "N8"         "N9"

N1 = Abuse or neglect
N2 = Child's disability/illness
N3 = Parental Disability/illness
N4 = Family in acute stress
N5 = Family dysfunction
N6 = Socially unacceptable
N7 = Low income
N8 = Absent parenting
N9 = Cases other than Children in Need
N0 = Not stated

In [8]:
sc_df <- sc_df |> 
    mutate(CIN_PrimaryNeedCode = str_trim(CIN_PrimaryNeedCode))

In [9]:
sc_df <- sc_df |> 
    mutate(CIN_PrimaryNeedCode = case_when(
               CIN_PrimaryNeedCode == 'n5' ~ 'N5',
               TRUE ~ CIN_PrimaryNeedCode
            ))

In [10]:
sc_df |> distinct(CIN_PrimaryNeedCode) |> pull(CIN_PrimaryNeedCode)

[1] "N5" "N1" "N8" "N4" NA   "N0" "N2" "N3" "N6" "N7" "N9"

In [11]:
sc_filtered <- sc_df |>
    filter(person_id %in% person_ids$person_id) |>
    select(person_id,CIN_ACADYR,CIN_PrimaryNeedCode)

In [12]:
sc_filtered |> group_by(CIN_PrimaryNeedCode) |> tally()

CIN_PrimaryNeedCode,n
<chr>,<int>
N0,376
N1,8315
N2,885
N3,136
N4,922
N5,1419
N6,203
N7,25
N8,106


In [13]:
sc_filtered |> nrow()

[1] 12618

In [14]:
sc_filtered |> 
    summarize(n_distinct(person_id))

n_distinct(person_id)
<int>
3652


In [15]:
sc_merge <- sc_filtered |>
    select(-CIN_ACADYR) |>
    distinct()

In [16]:
sc_merge |> nrow()

[1] 4770

In [17]:
sc_merge |> 
    summarize(n_distinct(person_id))

n_distinct(person_id)
<int>
3652


In [18]:
sc_merge |>
    group_by(person_id) |>
    filter(n() > 1) |>
    arrange(person_id)

person_id,CIN_PrimaryNeedCode
<chr>,<chr>
0045ADA48CA299BC4445318E8D951610DCB89A829665CFC821E92340F0F8B30D,NA
0045ADA48CA299BC4445318E8D951610DCB89A829665CFC821E92340F0F8B30D,N1
0093549A2B3FB44847A7880014804DB333707601AE77A4AF949EE71E538194D6,NA
0093549A2B3FB44847A7880014804DB333707601AE77A4AF949EE71E538194D6,N1
0093549A2B3FB44847A7880014804DB333707601AE77A4AF949EE71E538194D6,N2
0093549A2B3FB44847A7880014804DB333707601AE77A4AF949EE71E538194D6,N4
0093549A2B3FB44847A7880014804DB333707601AE77A4AF949EE71E538194D6,N5
017BD83D33E24C2CE047BFB67D7D010694F10CAF3E02B4360C88D4EE29E119FD,N1
017BD83D33E24C2CE047BFB67D7D010694F10CAF3E02B4360C88D4EE29E119FD,N4


In [19]:
ever_cin <- sc_merge |>
    select(person_id) |>
    distinct() |>
    mutate(ever_CIN = TRUE)

In [20]:
ever_cin

person_id,ever_CIN
<chr>,<lgl>
AAF20E0D9D31684973B137A74028F54A25D0C5EE563478BCB3702047DB25B55B,TRUE
22A611BACA79474CB520C45004E70419C711DD1AF450F034DD0A46C1417DA708,TRUE
683466122ED7D987E396F81D5EB4A3F7BECBA4F8D31DE8C1CD3208005CC6741B,TRUE
994DEBAAD8CF351CE9C31B89A669974A64F7BB43F315344C3EF31D83CC7CB92D,TRUE
2DA907C18DD6C9C21DEA87DB2807F96B4D720850953AA25E7EE3306457C6DA9E,TRUE
F5FD9F46523BDF5820C10C5C92C480B112402A40A9E1324915307C3BCACD5FEB,TRUE
2B8CCDD7863772F857002F23838E6D99BFC61CE8C8985867D1A4E007CD951A17,TRUE
3920D247A0AF6587F5E230C2BB6623D73088D83EB8B46F9CE9A902D95C04DBA7,TRUE
A1C8D9F7084F25EDA5B2B67FF73C5FEB419898D875CC597E5EA9D530C71B2DC2,TRUE


In [33]:
lac_extra

person_id,ever_CIN
<chr>,<lgl>
765802B6591B46CCCEFCDE9D32D960FD41C6A1A49207E9B6AFC4D8BAA8789F84,TRUE
21922CBFE180E8946C89A59782AC1E466C80D93E85008DE64189761B1A6E8D8C,TRUE
E2C4A8EAB1EFA92CFEFD8D2E6A091B122252B209CF2B99BA0CC72C93710DEC9C,TRUE
82762AA35A1AB1C1175FC7956C1E03E701C55462E5BA0E49714FB47C7D05EB9C,TRUE
59C4DF19BF86AC5D32BFC710E7A919531628BB507F8024E1BFECD192F6B569EE,TRUE
D21CC15BDFE3DC41EAD144257AA1A7B8A96E6DD90DACA1F5F7C6F1F17014CF96,TRUE
5F1AD58EC267D213DF017826D51B3C9544777C6F9197ED6E2053D428A4AA7E33,TRUE
B6275FAD8102F90F9E123AA0D524DFEF471A00E201B68A9CA9EBA13E290DEAE9,TRUE
AF5F6E07B22B1FEC1F64D15E42C2FE07724BC528BA4CF668DF72AD62B7E01F51,TRUE


In [34]:
ever_cin <- rbind(ever_cin,lac_extra)

In [35]:
ever_cin |> nrow()

[1] 3662

In [36]:
head(ever_cin)

person_id,ever_CIN
<chr>,<lgl>
AAF20E0D9D31684973B137A74028F54A25D0C5EE563478BCB3702047DB25B55B,TRUE
22A611BACA79474CB520C45004E70419C711DD1AF450F034DD0A46C1417DA708,TRUE
683466122ED7D987E396F81D5EB4A3F7BECBA4F8D31DE8C1CD3208005CC6741B,TRUE
994DEBAAD8CF351CE9C31B89A669974A64F7BB43F315344C3EF31D83CC7CB92D,TRUE
2DA907C18DD6C9C21DEA87DB2807F96B4D720850953AA25E7EE3306457C6DA9E,TRUE
F5FD9F46523BDF5820C10C5C92C480B112402A40A9E1324915307C3BCACD5FEB,TRUE


### Save csv

In [37]:
# save as csv
write.csv(ever_cin, "data/additional_ever_cin.csv", row.names = FALSE)

# CIN Disability 2009-2019

In [14]:
dis_data = "CB_2489.cb_CIN_2009_to_2019_Disability"

dis_table <- tbl(con, dis_data) |>
    select(person_id,CIN_ACADYR,CIN_Disability) 

In [27]:
dis_df <- collect(dis_table)

In [28]:
head(dis_df)

person_id,CIN_ACADYR,CIN_Disability
<chr>,<chr>,<chr>
EDA994097F3DDFA87825365C60773712D04CD3654F794D63C55D9E3932D484E0,2008/2009,NA
DBD4209B51C4884D3BCCBA99F753B1B09189BF502241237C59D178D47810FF0A,2008/2009,NA
DAC8EEC48D224CBDDAEA16BB114F1364011C78E6C1B6EF7C8941A3E231F91B16,2008/2009,NA
49C7AB9FEB601D00D97784F5B4B91DAD30B87E65AA9F6777B1FC920EA3AFD43C,2008/2009,NA
54CC7198D9E1CDB2182E47800199B7FA36FADF3695B08BC20DC389F69C703D93,2008/2009,NA
9BBCB3B30E354EE0BBD65262E0DBA7B7654CFA72E3E1643324CE5F60C4FCBDE6,2008/2009,NA


In [24]:
dis_df |> arrange(CIN_ACADYR) |> distinct(CIN_ACADYR) |> pull(CIN_ACADYR)

[1] "2008/2009" "2009/2010" "2010/2011" "2011/2012" "2012/2013" "2013/2014"
 [7] "2014/2015" "2015/2016" "2016/2017" "2017/2018" "2018/2019"

In [25]:
dis_df |> distinct(CIN_Disability) |> pull(CIN_Disability)

[1] NA     "LD"   "PC"   "AUT"  "BEH"  "CON"  "DDA"  "INC"  "MOB"  "VIS" 
[11] "COMM" "HAND" "HEAR" "NONE" "None" "none"

Holds a record of the type of disability(s) a child may suffer from. NONE by itself is used for no disability.

NONE = None
MOB = Mobility
HAND = Hand Function
PC = Personal Care
INC = Incontinence
COMM = Communication
LD = Learning
HEAR = Hearing
VIS = Vision
BEH = Behaviour
CON = Consciousness
AUT = Diagnosed with autism or Aspergers syndrome 
DDA = Disabled under DDA but not in above categories

In [29]:
dis_tidy <- dis_df |> 
    mutate(CIN_Disability = case_when(
        is.na(CIN_Disability) | CIN_Disability %in% c('NONE', 'None', 'none') ~ NA_character_,
        TRUE ~ CIN_Disability
    ))

In [30]:
dis_tidy |> group_by(CIN_Disability) |> tally()

CIN_Disability,n
<chr>,<int>
AUT,1980
BEH,2495
COMM,2878
CON,839
DDA,754
HAND,1064
HEAR,701
INC,1581
LD,4084


In [31]:
dis_filtered <- dis_tidy |>
    filter(person_id %in% person_ids$person_id)

In [32]:
dis_filtered |> group_by(CIN_Disability) |> tally()

CIN_Disability,n
<chr>,<int>
AUT,234
BEH,330
COMM,359
CON,96
DDA,119
HAND,129
HEAR,111
INC,278
LD,436


# SS CIN 

In [33]:
cin_data = "CB_2489.tbl_SocialServices_CiNP"

cin_table <- tbl(con, cin_data) |>
    select(person_id,StartDate,EndDate) 

In [34]:
cin_df <- collect(cin_table)

In [35]:
head(cin_df)

person_id,StartDate,EndDate
<chr>,<chr>,<chr>
5356366A24004B6B14DAAAEE9ADD3EFC661545E444D05C5D4C6001494A3CC372,01/04/2019,
D6B3B45878368C7D67E405D9F608352B0C35D272A9BB8A31DFC350445C58CE91,01/04/2019,27/08/2020
63D51806BF39CD8BA75B167A225A6265CACB64C2BCC450D1CC6A8D1B7762C51A,01/04/2019,28/09/2019
E891B044F3EF5574592575824696FCB0A36088AC566895C92256707227EFE7B4,01/04/2019,30/05/2019
FE31EDBA16EA714C02AF362D2D38E6845D1F61EDA058E3C1FBED9BB816287D4D,01/05/2019,04/07/2019
88B89CC5F7F97D31198DDA1EB490F4726DCD596D645BED3601D35F50530B9B14,01/05/2019,04/07/2019


In [36]:
cin_df |> arrange(StartDate) |> distinct(StartDate) |> pull(StartDate)

[1] "01/02/2021" "01/03/2020" "01/03/2021" "01/04/2019" "01/04/2020"
  [6] "01/04/2021" "01/05/2019" "01/05/2020" "01/06/2020" "01/06/2021"
 [11] "01/07/2019" "01/08/2019" "01/09/2020" "01/10/2019" "01/10/2020"
 [16] "01/11/2019" "01/11/2020" "01/12/2020" "02/01/2020" "02/02/2021"
 [21] "02/03/2020" "02/03/2021" "02/04/2019" "02/04/2020" "02/05/2019"
 [26] "02/05/2021" "02/06/2020" "02/06/2021" "02/07/2019" "02/07/2020"
 [31] "02/08/2019" "02/09/2019" "02/09/2020" "02/10/2019" "02/10/2020"
 [36] "02/11/2020" "02/12/2019" "02/12/2020" "03/01/2020" "03/01/2021"
 [41] "03/02/2020" "03/02/2021" "03/03/2020" "03/03/2021" "03/04/2019"
 [46] "03/04/2020" "03/04/2021" "03/05/2019" "03/05/2021" "03/06/2019"
 [51] "03/06/2020" "03/06/2021" "03/07/2019" "03/07/2020" "03/08/2020"
 [56] "03/09/2019" "03/09/2020" "03/10/2019" "03/11/2019" "03/11/2020"
 [61] "03/12/2019" "03/12/2020" "04/01/2021" "04/02/2020" "04/02/2021"
 [66] "04/03/2020" "04/03/2021" "04/04/2019" "04/05/2020" "04/05/2021"
 [71] "04/06/2019" "04/06/2020" "04/06/2021" "04/07/2019" "04/08/2020"
 [76] "04/09/2019" "04/09/2020" "04/10/2019" "04/11/2019" "04/11/2020"
 [81] "04/12/2019" "04/12/2020" "05/01/2021" "05/02/2020" "05/02/2021"
 [86] "05/03/2020" "05/03/2021" "05/04/2019" "05/05/2020" "05/05/2021"
 [91] "05/06/2019" "05/06/2020" "05/07/2019" "05/08/2019" "05/08/2020"
 [96] "05/09/2019" "05/10/2020" "05/11/2019" "05/11/2020" "05/12/2019"
[101] "06/01/2020" "06/01/2021" "06/02/2020" "06/03/2020" "06/04/2020"
[106] "06/04/2021" "06/05/2019" "06/05/2020" "06/05/2021" "06/06/2019"
[111] "06/07/2020" "06/08/2019" "06/08/2020" "06/09/2019" "06/09/2020"
[116] "06/10/2019" "06/10/2020" "06/11/2019" "06/11/2020" "06/12/2019"
[121] "07/01/2020" "07/01/2021" "07/02/2020" "07/04/2020" "07/04/2021"
[126] "07/05/2019" "07/05/2020" "07/05/2021" "07/06/2021" "07/07/2019"
[131] "07/07/2020" "07/08/2019" "07/09/2020" "07/10/2019" "07/10/2020"
[136] "07/12/2020" "08/01/2020" "08/01/2021" "08/02/2021" "08/03/2021"
[141] "08/04/2019" "08/04/2020" "08/04/2021" "08/05/2019" "08/05/2020"
[146] "08/06/2020" "08/06/2021" "08/07/2019" "08/07/2020" "08/08/2019"
[151] "08/09/2020" "08/10/2019" "08/10/2020" "08/11/2019" "08/11/2020"
[156] "08/12/2020" "09/01/2020" "09/02/2021" "09/03/2020" "09/03/2021"
[161] "09/04/2019" "09/04/2020" "09/04/2021" "09/05/2019" "09/05/2021"
[166] "09/06/2020" "09/06/2021" "09/07/2019" "09/07/2020" "09/08/2019"
[171] "09/09/2019" "09/09/2020" "09/10/2019" "09/10/2020" "09/11/2019"
[176] "09/11/2020" "09/12/2019" "09/12/2020" "10/01/2020" "10/02/2020"
[181] "10/02/2021" "10/03/2020" "10/03/2021" "10/04/2019" "10/05/2019"
[186] "10/05/2021" "10/06/2019" "10/06/2020" "10/07/2019" "10/07/2020"
[191] "10/08/2020" "10/09/2019" "10/09/2020" "10/10/2019" "10/11/2020"
[196] "10/12/2019" "10/12/2020" "11/01/2021" "11/02/2020" "11/02/2021"
[201] "11/03/2020" "11/03/2021" "11/04/2019" "11/04/2021" "11/05/2019"
[206] "11/05/2020" "11/05/2021" "11/06/2019" "11/06/2020" "11/07/2019"
[211] "11/08/2020" "11/09/2019" "11/09/2020" "11/10/2019" "11/11/2019"
[216] "11/11/2020" "11/12/2019" "11/12/2020" "12/01/2021" "12/02/2020"
[221] "12/02/2021" "12/03/2020" "12/03/2021" "12/04/2019" "12/04/2021"
[226] "12/05/2019" "12/05/2020" "12/05/2021" "12/06/2019" "12/07/2019"
[231] "12/08/2020" "12/09/2019" "12/10/2020" "12/11/2019" "12/11/2020"
[236] "12/12/2019" "13/01/2020" "13/01/2021" "13/02/2020" "13/02/2021"
[241] "13/03/2020" "13/04/2019" "13/04/2020" "13/04/2021" "13/05/2019"
[246] "13/05/2020" "13/05/2021" "13/06/2019" "13/07/2019" "13/07/2020"
[251] "13/08/2019" "13/08/2020" "13/09/2019" "13/09/2020" "13/10/2019"
[256] "13/10/2020" "13/11/2019" "13/11/2020" "13/12/2019" "14/01/2020"
[261] "14/01/2021" "14/02/2020" "14/04/2019" "14/04/2020" "14/04/2021"
[266] "14/05/2019" "14/05/2020" "14/05/2021" "14/06/2019" "14/07/2020"
[271] "14/08/2019" "14/08/2020" "14/09/2020" "14/10/2019" "14/10/2020"
[276] "14/11/2019" "14/12/2020" "15/01/2020" "15/01/2021" "15/02/2020"
[281] "15/02/2021" "15/03/

# LAC

In [22]:
lac_data = "CB_2489.cb_CLA_2006_to_2019"

lac_table <- tbl(con, lac_data) |>
    select(person_id,CLA_ACADYR) 

In [23]:
lac_df <- collect(lac_table)

In [24]:
head(lac_df)

person_id,CLA_ACADYR
<chr>,<chr>
301A8393CBC9BC5B127B33C9BBC77D89A67932C531AB716D0AC116A29FBC03A7,2005/2006
4C7F526CAEFC48E30751765A0168DFFDF67E717B229E1FFAF0AAADEDA18A3773,2005/2006
243542528EE5BD4E50EEDE9AC7C9587633F5275D27AC69975A43E4634375A484,2005/2006
FF008416E57E330193805A59438F4983E43547DFFF02A162BB31742F5AACEA4D,2005/2006
B93124CA7B49C443CE539682B23536D8F1EB1D35609E1E730AD982CD120D45FB,2005/2006
EE4A1478A07E12766FF1D5DA01DF603A98A4ADE131718D071060F30142D49D3F,2005/2006


In [25]:
lac_df |> arrange(CLA_ACADYR) |> distinct(CLA_ACADYR) |> pull(CLA_ACADYR)

[1] "2005/2006" "2006/2007" "2007/2008" "2008/2009" "2009/2010" "2010/2011"
 [7] "2011/2012" "2012/2013" "2013/2014" "2014/2015" "2015/2016" "2016/2017"
[13] "2017/2018" "2018/2019"

In [26]:
lac_filtered <- lac_df |>
    filter(person_id %in% person_ids$person_id)

In [27]:
ever_lac <- lac_filtered |>
    select(person_id) |>
    distinct() |>
    mutate(ever_LAC = TRUE)

In [28]:
ever_lac |> nrow()

[1] 536

In [29]:
sum(ever_lac$person_id %in% ever_cin$person_id)

[1] 526

In [30]:
lac_extra <- anti_join(ever_lac, ever_cin, by = "person_id")

In [31]:
lac_extra

person_id,ever_LAC
<chr>,<lgl>
765802B6591B46CCCEFCDE9D32D960FD41C6A1A49207E9B6AFC4D8BAA8789F84,TRUE
21922CBFE180E8946C89A59782AC1E466C80D93E85008DE64189761B1A6E8D8C,TRUE
E2C4A8EAB1EFA92CFEFD8D2E6A091B122252B209CF2B99BA0CC72C93710DEC9C,TRUE
82762AA35A1AB1C1175FC7956C1E03E701C55462E5BA0E49714FB47C7D05EB9C,TRUE
59C4DF19BF86AC5D32BFC710E7A919531628BB507F8024E1BFECD192F6B569EE,TRUE
D21CC15BDFE3DC41EAD144257AA1A7B8A96E6DD90DACA1F5F7C6F1F17014CF96,TRUE
5F1AD58EC267D213DF017826D51B3C9544777C6F9197ED6E2053D428A4AA7E33,TRUE
B6275FAD8102F90F9E123AA0D524DFEF471A00E201B68A9CA9EBA13E290DEAE9,TRUE
AF5F6E07B22B1FEC1F64D15E42C2FE07724BC528BA4CF668DF72AD62B7E01F51,TRUE


10 extra are dates prior to CIN dataset

In [32]:
lac_extra <- lac_extra |>
    rename(ever_CIN = ever_LAC)